# 01 — Pair dataset audit and preparation

This notebook verifies the source tables, preserves the dataset author’s explicit relevant-pair membership, and writes the canonical handoff for Notebook 02. The data are synthetic. Match lists contain positive memberships only and are capped at 30 resumes per job; an unlisted pair is unlabeled, not a negative example.

In [ ]:
from pathlib import Path
import sys

# Find the project even when Jupyter starts in a child folder.
start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents] if (p / "manage.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside AI_RESUME_ANALYSIS or one of its subfolders.")
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.matching_data import write_positive_pairs


In [ ]:
pairs_path = write_positive_pairs(PROJECT_ROOT)
import pandas as pd
pairs = pd.read_csv(pairs_path)
manifest_path = PROJECT_ROOT / "data" / "raw" / "dataset_manifest.json"
manifest = __import__("json").loads(manifest_path.read_text(encoding="utf-8"))

required = {"resume_id", "job_id", "resume_text", "job_description", "match_label", "label_source"}
assert required.issubset(pairs.columns)
assert pairs[["resume_text", "job_description"]].notna().all().all()
assert set(pairs["match_label"]) == {"explicit_relevant"}
print("Dataset:", manifest["dataset_id"])
print("Pinned revision:", manifest["revision"])
print("Declared license:", manifest["license"])
print("Explicit relevant pairs:", len(pairs))
print("Unique resumes/jobs:", pairs.resume_id.nunique(), pairs.job_id.nunique())
print("Labels present:", sorted(pairs.match_label.unique()))
print("No negative or graded labels are present; none were created.")
pairs.head(3)
